# Sales Data Pipeline Using PySpark and Databricks

## Project Overview
This project builds a sales data pipeline using PySpark and Databricks.

## Objectives
- Create sample sales data.
- Store raw data in the Bronze layer.
- Clean and transform data in the Silver layer.
- Generate product-wise sales summaries in the Gold layer.
- Perform data quality checks.

## Technologies
- Python
- PySpark
- Databricks
- Delta Lake
- Spark SQL

## Pipeline Architecture
CSV Source → Bronze → Silver → Gold → Data Quality Checks

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

schema = StructType([
    StructField("Order_ID", IntegerType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True)
])

data = [
    (101, "Laptop", 2, 50000.0),
    (102, "Mouse", 5, 500.0),
    (103, "Keyboard", 3, 1500.0),
    (104, "Monitor", 2, 12000.0),
    (105, "Laptop", 1, 50000.0),
    (106, "Mouse", 4, 500.0),
    (107, "Keyboard", 2, 1500.0),
    (108, "Monitor", 3, 12000.0)
]

sales_df = spark.createDataFrame(data, schema)

display(sales_df)

Order_ID,Product,Quantity,Price
101,Laptop,2,50000.0
102,Mouse,5,500.0
103,Keyboard,3,1500.0
104,Monitor,2,12000.0
105,Laptop,1,50000.0
106,Mouse,4,500.0
107,Keyboard,2,1500.0
108,Monitor,3,12000.0


## 1. Bronze Layer – Raw Data

The Bronze layer stores the original data with minimal changes.

In [0]:
sales_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.bronze_sales")

bronze_df = spark.table("default.bronze_sales")

display(bronze_df)

Order_ID,Product,Quantity,Price
101,Laptop,2,50000.0
102,Mouse,5,500.0
103,Keyboard,3,1500.0
104,Monitor,2,12000.0
105,Laptop,1,50000.0
106,Mouse,4,500.0
107,Keyboard,2,1500.0
108,Monitor,3,12000.0


## 2. Silver Layer – Cleaned Data

The Silver layer removes duplicate records, handles invalid values,
and calculates total sales for each order.

In [0]:
from pyspark.sql.functions import col

silver_df = (
    bronze_df
    .dropDuplicates(["Order_ID"])
    .filter(
        col("Order_ID").isNotNull() &
        col("Product").isNotNull() &
        col("Quantity").isNotNull() &
        col("Price").isNotNull() &
        (col("Quantity") > 0) &
        (col("Price") >= 0)
    )
    .withColumn(
        "Total_Sales",
        col("Quantity") * col("Price")
    )
)

silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.silver_sales")

silver_df = spark.table("default.silver_sales")

display(silver_df)

Order_ID,Product,Quantity,Price,Total_Sales
105,Laptop,1,50000.0,50000.0
104,Monitor,2,12000.0,24000.0
108,Monitor,3,12000.0,36000.0
106,Mouse,4,500.0,2000.0
103,Keyboard,3,1500.0,4500.0
107,Keyboard,2,1500.0,3000.0
102,Mouse,5,500.0,2500.0
101,Laptop,2,50000.0,100000.0


## 3. Gold Layer – Sales Summary

The Gold layer contains aggregated data for reporting and analysis.
It calculates total quantity, total revenue, and number of orders
for each product.

In [0]:
from pyspark.sql.functions import sum, count

gold_df = (
    silver_df
    .groupBy("Product")
    .agg(
        sum("Quantity").alias("Total_Quantity"),
        sum("Total_Sales").alias("Total_Revenue"),
        count("Order_ID").alias("Number_of_Orders")
    )
    .orderBy("Product")
)

gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.gold_sales_summary")

gold_df = spark.table("default.gold_sales_summary")

display(gold_df)

Product,Total_Quantity,Total_Revenue,Number_of_Orders
Keyboard,5,7500.0,2
Laptop,3,150000.0,2
Monitor,5,60000.0,2
Mouse,9,4500.0,2


## 4. Data Quality Checks

Data quality checks verify that the Silver layer contains valid records.
We check for duplicate order IDs, null values, and invalid quantities.

In [0]:
from pyspark.sql.functions import col

duplicate_orders = (
    silver_df.groupBy("Order_ID")
    .count()
    .filter(col("count") > 1)
    .count()
)

null_orders = silver_df.filter(
    col("Order_ID").isNull() |
    col("Product").isNull() |
    col("Quantity").isNull() |
    col("Price").isNull() |
    col("Total_Sales").isNull()
).count()

invalid_orders = silver_df.filter(
    (col("Quantity") <= 0) |
    (col("Price") < 0)
).count()

print("Duplicate Order IDs:", duplicate_orders)
print("Rows with Null Values:", null_orders)
print("Rows with Invalid Values:", invalid_orders)

Duplicate Order IDs: 0
Rows with Null Values: 0
Rows with Invalid Values: 0


## 5. Final Results

The sales data pipeline has three layers:

- Bronze: Stores raw sales data.
- Silver: Stores cleaned data with calculated sales.
- Gold: Stores product-wise sales summaries.

Data quality checks help verify the cleaned data.
The Gold layer can be used for reporting and business analysis.

In [0]:
print("BRONZE LAYER")
display(spark.table("default.bronze_sales"))

print("SILVER LAYER")
display(spark.table("default.silver_sales"))

print("GOLD LAYER")
display(spark.table("default.gold_sales_summary"))

BRONZE LAYER


Order_ID,Product,Quantity,Price
101,Laptop,2,50000.0
102,Mouse,5,500.0
103,Keyboard,3,1500.0
104,Monitor,2,12000.0
105,Laptop,1,50000.0
106,Mouse,4,500.0
107,Keyboard,2,1500.0
108,Monitor,3,12000.0


SILVER LAYER


Order_ID,Product,Quantity,Price,Total_Sales
105,Laptop,1,50000.0,50000.0
104,Monitor,2,12000.0,24000.0
108,Monitor,3,12000.0,36000.0
106,Mouse,4,500.0,2000.0
103,Keyboard,3,1500.0,4500.0
107,Keyboard,2,1500.0,3000.0
102,Mouse,5,500.0,2500.0
101,Laptop,2,50000.0,100000.0


GOLD LAYER


Product,Total_Quantity,Total_Revenue,Number_of_Orders
Keyboard,5,7500.0,2
Laptop,3,150000.0,2
Monitor,5,60000.0,2
Mouse,9,4500.0,2


## 6. Final Validation

Verify the total number of orders, products, and revenue.

In [0]:
from pyspark.sql.functions import sum, count, countDistinct

silver_df = spark.table("default.silver_sales")

silver_df.select(
    "Order_ID", "Product", "Quantity", "Price", "Total_Sales"
).show()

silver_df.agg(
    count("Order_ID").alias("Total_Orders"),
    countDistinct("Product").alias("Total_Products"),
    sum("Total_Sales").alias("Total_Revenue")
).show()

+--------+--------+--------+-------+-----------+
|Order_ID| Product|Quantity|  Price|Total_Sales|
+--------+--------+--------+-------+-----------+
|     105|  Laptop|       1|50000.0|    50000.0|
|     104| Monitor|       2|12000.0|    24000.0|
|     108| Monitor|       3|12000.0|    36000.0|
|     106|   Mouse|       4|  500.0|     2000.0|
|     103|Keyboard|       3| 1500.0|     4500.0|
|     107|Keyboard|       2| 1500.0|     3000.0|
|     102|   Mouse|       5|  500.0|     2500.0|
|     101|  Laptop|       2|50000.0|   100000.0|
+--------+--------+--------+-------+-----------+

+------------+--------------+-------------+
|Total_Orders|Total_Products|Total_Revenue|
+------------+--------------+-------------+
|           8|             4|     222000.0|
+------------+--------------+-------------+

